In [0]:
import pyspark.sql.functions as F 
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/tosinforlly@gmail.com/fmcg_project/1_setup/utilities

In [0]:
# Widgets Activation
dbutils.widgets.text('catalog', 'spotizone', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

# Path Variables
storage_path = f's3://sport-bar/{data_source}'
landing_path = f'{storage_path}/landing/'
processed_path = f'{storage_path}/processed/'

# Table Variables
bronze = f'{catalog}.{bronze_schema}.{data_source}'
silver = f'{catalog}.{silver_schema}.{data_source}'
gold = f'{catalog}.{gold_schema}.sb_dim{data_source}'

#### Bronze Layer

In [0]:
df_ord = (
    spark.read.format("csv")
    .option("inferSchema", True)
    .option("header", True)
    .load(landing_path)
    .withColumn("ingest_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name")
)

In [0]:
display(df_ord)

In [0]:
df_ord.write\
    .option('delta.enableChangeDataFeed', 'true')\
    .mode("append")\
    .format("delta")\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

In [0]:
files = dbutils.fs.ls(landing_path)

for fi in files:
    dbutils.fs.mv(
        fi.path, f'{processed_path}/{fi.name}',
        True
    )

#### SILVER LAYER

In [0]:
df_bronze = spark.table(f'{catalog}.{bronze_schema}.{data_source}')

In [0]:
df_silver = df_bronze
display(df_silver)

##### Data Tranformation


- 1. Removing NULLS in order_id if preent

In [0]:
df_silver = df_silver.filter(F.col('order_id').isNotNull())

- 2. Clean custormer_id for Consistent Numeric Values

In [0]:
df_silver = df_silver.withColumn(
  'customer_id',
  F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id').cast('string'))
  .otherwise(F.lit('999999').cast('string')
))

In [0]:
display(df_silver.limit(50))

- 3. order_placement_date Formatting

In [0]:
df_silver = df_silver.withColumn(
  "order_placement_date", F.regexp_replace(F.col("order_placement_date"), "^[A-Za-z]+,\\s*", "")
  )

# Parse all date format in order_placement_date
df_silver = (
  df_silver.withColumn(
    'order_placement_date',
      F.coalesce(
        F.try_to_date(F.col('order_placement_date'), 'yyyy/MM/dd'),
        F.try_to_date(F.col('order_placement_date'), 'dd-MM-yyyy'),
        F.try_to_date(F.col('order_placement_date'), 'MMMM dd, yyyy'),
        F.try_to_date(F.col('order_placement_date'), 'dd/MM/yyyy')
      )
    )
)

In [0]:
display(df_silver.limit(10))

- 4. Drop Null order_qty and Duplicates Where all Columns are matching

In [0]:
df_silver = (
  df_silver\
    .filter(F.col('order_qty').isNotNull())\
    .dropDuplicates([
      'order_id',
      'order_placement_date',
      'customer_id',
      'product_id',
      'order_qty'])\
    .withColumn('product_id', F.col('product_id').cast('string'))
)
display(df_silver.limit(10))

- 5. Join Product code from silver.product

In [0]:
prod = spark.table('spotizone.silver.products')

# Join
df_silver = df_silver.join(
    prod, 'product_id', 'inner')\
    .select(
        df_silver['*'], prod['product_code'])

In [0]:
display(df_silver.limit(10))

##### Write to silver layer

In [0]:
df_silver\
    .write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

#### GOLD LAYER

In [0]:
df_gold = spark.sql(f'SELECT order_id, order_placement_date AS date, customer_id AS customer_code, product_id, product_code, order_qty AS sold_quantity FROM {silver}')

display(df_gold.limit(10))

In [0]:
df_gold\
    .write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f'{catalog}.{gold_schema}.sb_fact_{data_source}')

##### Merge With Parent Company

In [0]:
df_child = spark.table(('spotizone.gold.sb_fact_orders'))

df_child = df_child.select(['date', 'product_code', 'customer_code', 'sold_quantity'])
display(df_child.limit(10))

In [0]:
# Date attribute is aggregated by start_month in parent table, so transform child date column
df_child = df_child\
    .withColumn('date', F.trunc('date', 'MM'))\
    .groupBy('date', 'product_code', 'customer_code')\
    .agg(F.sum('sold_quantity').alias('sold_quantity'))

In [0]:
parent = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.fact_orders')

parent.alias('target')\
    .merge(df_child.alias('source'),
           'target.date = source.date AND target.product_code = source.product_code AND target.customer_code = source.customer_code AND target.sold_quantity = source.sold_quantity')\
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:
display(df_child.limit(10))